In [ ]:
# ============================================================
# BLUESTOCK MUTUAL FUND ANALYTICS
# DAY 4 - PERFORMANCE ANALYTICS
# ============================================================

Step 1 — Load Data

Since database already exists:

In [6]:
import sqlite3
import pandas as pd
import numpy as np

# Connect database
conn = sqlite3.connect("../data/db/bluestock_mf.db")

# Load tables
funds = pd.read_sql(
    "SELECT * FROM dim_fund",
    conn
)

nav = pd.read_sql(
    "SELECT * FROM fact_nav",
    conn
)

perf = pd.read_sql(
    "SELECT * FROM fact_performance",
    conn
)

# Convert date column
nav["nav_date"] = pd.to_datetime(nav["nav_date"])

# Sort data
nav = nav.sort_values(
    ["amfi_code", "nav_date"]
)

print("Funds Shape :", funds.shape)
print("NAV Shape   :", nav.shape)
print("Perf Shape  :", perf.shape)

nav.head()

Funds Shape : (40, 13)
NAV Shape   : (46000, 5)
Perf Shape  : (40, 14)


,nav_id,amfi_code,nav_date,nav,daily_return_pct
0,1,100016,2022-01-03,520.4608,NaN
1,2,100016,2022-01-04,515.0971,-1.030568
2,3,100016,2022-01-05,521.7239,1.286515
3,4,100016,2022-01-06,515.7880,-1.137747
4,5,100016,2022-01-07,515.1639,-0.120999


In [7]:
annual_returns = (
    nav.groupby("amfi_code")["daily_return_pct"]
       .apply(
           lambda x: (
               (1 + x.dropna()).prod()
           ) ** (252 / len(x.dropna())) - 1
       )
       .reset_index(name="annualized_return")
)

annual_returns.head()

C:\Users\SUJAL SARNOBAT\AppData\Local\Temp\ipykernel_25712\1324275665.py:4: RuntimeWarning: invalid value encountered in scalar power
  lambda x: (


,amfi_code,annualized_return
0,100016,-1.000000
1,100025,-0.983963
2,100033,NaN
3,101206,NaN
4,101207,NaN


In [8]:
annual_returns["annualized_return_pct"] = (
    annual_returns["annualized_return"] * 100
)

annual_returns.head()

,amfi_code,annualized_return,annualized_return_pct
0,100016,-1.000000,-100.00000
1,100025,-0.983963,-98.39628
2,100033,NaN,NaN
3,101206,NaN,NaN
4,101207,NaN,NaN


In [9]:
top10_annual = (
    annual_returns
    .sort_values(
        "annualized_return_pct",
        ascending=False
    )
    .head(10)
)

top10_annual

,amfi_code,annualized_return,annualized_return_pct
22,119599,12904.988183,1.290499e+06
27,120507,691.538630,6.915386e+04
31,120844,515.911000,5.159110e+04
5,101208,361.196614,3.611966e+04
17,119095,9.934703,9.934703e+02
18,119120,-0.943424,-9.434242e+01
13,118636,-0.971246,-9.712460e+01
1,100025,-0.983963,-9.839628e+01
16,119094,-1.000000,-1.000000e+02
36,148569,-1.000000,-1.000000e+02


In [10]:
top10_annual = top10_annual.merge(
    funds[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

top10_annual[
    ["scheme_name", "annualized_return_pct"]
]

,scheme_name,annualized_return_pct
0,SBI Small Cap Fund - Direct Plan - Growth,1.290499e+06
1,ICICI Pru Liquid Fund - Regular - Growth,6.915386e+04
2,Kotak Liquid Fund - Regular - Growth,5.159110e+04
3,ABSL Liquid Fund - Regular - Growth,3.611966e+04
4,Axis Small Cap Fund - Regular - Growth,9.934703e+02
5,SBI Magnum Gilt Fund - Regular Plan - Growth,-9.434242e+01
6,Nippon India Gilt Securities Fund - Regular - ...,-9.712460e+01
7,HDFC Short Term Debt Fund - Regular - Growth,-9.839628e+01
8,Axis Midcap Fund - Regular - Growth,-1.000000e+02
9,Mirae Asset Tax Saver Fund - Regular - Growth,-1.000000e+02


In [11]:
import plotly.express as px

fig = px.bar(
    top10_annual,
    x="annualized_return_pct",
    y="scheme_name",
    orientation="h",
    title="Top 10 Funds by Annualized Return (%)"
)

fig.show()

In [12]:
annual_returns.to_csv(
    "../reports/annualized_returns.csv",
    index=False
)

Step 3 — CAGR Calculation (1Y, 3Y, 5Y)

CAGR (Compound Annual Growth Rate) tells us the annualized growth of a fund over a specific period.

In [14]:
def calculate_cagr(fund_df, years):

    fund_df = fund_df.sort_values("nav_date")

    end_date = fund_df["nav_date"].max()
    start_date = end_date - pd.DateOffset(years=years)

    period_data = fund_df[
        fund_df["nav_date"] >= start_date
    ]

    if len(period_data) < 2:
        return np.nan

    start_nav = period_data.iloc[0]["nav"]
    end_nav = period_data.iloc[-1]["nav"]

    cagr = (
        (end_nav / start_nav) ** (1 / years)
    ) - 1

    return cagr

In [15]:
cagr_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    cagr_results.append({

        "amfi_code": amfi_code,

        "cagr_1yr":
            calculate_cagr(group, 1),

        "cagr_3yr":
            calculate_cagr(group, 3),

        "cagr_5yr":
            calculate_cagr(group, 5)

    })

cagr_df = pd.DataFrame(cagr_results)

In [16]:
for col in [
    "cagr_1yr",
    "cagr_3yr",
    "cagr_5yr"
]:
    cagr_df[col] *= 100

cagr_df.head()

,amfi_code,cagr_1yr,cagr_3yr,cagr_5yr
0,100016,-2.224271,1.292649,2.316843
1,100025,3.704969,3.916390,3.912653
2,100033,53.232396,32.442459,26.074068
3,101206,47.924120,28.967695,20.442730
4,101207,-23.986032,-4.152381,6.953336


In [17]:
cagr_df = cagr_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

cagr_df.head()

,amfi_code,cagr_1yr,cagr_3yr,cagr_5yr,scheme_name
0,100016,-2.224271,1.292649,2.316843,HDFC Top 100 Fund - Regular Plan - Growth
1,100025,3.704969,3.916390,3.912653,HDFC Short Term Debt Fund - Regular - Growth
2,100033,53.232396,32.442459,26.074068,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
3,101206,47.924120,28.967695,20.442730,ABSL Frontline Equity Fund - Regular - Growth
4,101207,-23.986032,-4.152381,6.953336,ABSL Small Cap Fund - Regular - Growth


In [18]:
top10_cagr = (
    cagr_df
    .sort_values(
        "cagr_3yr",
        ascending=False
    )
    .head(10)
)

top10_cagr[
    ["scheme_name", "cagr_3yr"]
]

,scheme_name,cagr_3yr
16,Axis Midcap Fund - Regular - Growth,35.111802
34,Mirae Asset Large Cap Fund - Regular - Growth,34.000916
24,ICICI Pru Bluechip Fund - Direct - Growth,32.487429
2,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,32.442459
25,ICICI Pru Midcap Fund - Regular - Growth,31.777537
19,SBI Bluechip Fund - Regular Plan - Growth,30.456524
30,Kotak Flexicap Fund - Regular - Growth,29.582770
36,Mirae Asset Tax Saver Fund - Regular - Growth,29.178902
3,ABSL Frontline Equity Fund - Regular - Growth,28.967695
39,DSP Small Cap Fund - Regular - Growth,27.000427


In [19]:
import plotly.express as px

fig = px.bar(
    top10_cagr,
    x="cagr_3yr",
    y="scheme_name",
    orientation="h",
    title="Top 10 Funds by 3-Year CAGR (%)"
)

fig.show()

In [20]:
cagr_df.to_csv(
    "../reports/cagr_report.csv",
    index=False
)

In [21]:
cagr_df[
    ["cagr_1yr","cagr_3yr","cagr_5yr"]
].describe()

,cagr_1yr,cagr_3yr,cagr_5yr
count,40.000000,40.000000,40.000000
mean,19.428520,16.414715,14.541077
std,22.912276,12.206752,8.901844
min,-42.797615,-11.705807,1.030350
25%,7.377949,6.600925,6.013784
50%,17.474125,18.233102,14.476118
75%,27.161651,26.902600,21.257057
max,82.776059,35.111802,28.376762


Step 4 — Sharpe Ratio
What it measures

Sharpe Ratio tells us:

"How much return is generated for each unit of risk taken?"

Higher Sharpe = Better risk-adjusted performance.

In [22]:
RF = 0.065

sharpe_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    returns = group["daily_return_pct"].dropna()

    if len(returns) < 30:
        continue

    annual_return = returns.mean() * 252

    annual_volatility = (
        returns.std() * np.sqrt(252)
    )

    sharpe = (
        annual_return - RF
    ) / annual_volatility

    sharpe_results.append({

        "amfi_code": amfi_code,

        "annual_return": annual_return,

        "annual_volatility": annual_volatility,

        "sharpe_ratio": sharpe

    })

sharpe_df = pd.DataFrame(sharpe_results)

In [23]:
sharpe_df = sharpe_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

sharpe_df.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name
0,100016,3.568303,14.548135,0.240808,HDFC Top 100 Fund - Regular Plan - Growth
1,100025,4.285356,3.905246,1.080689,HDFC Short Term Debt Fund - Regular - Growth
2,100033,27.211057,18.936711,1.433515,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
3,101206,21.464662,14.568213,1.468928,ABSL Frontline Equity Fund - Regular - Growth
4,101207,10.696212,25.797322,0.412105,ABSL Small Cap Fund - Regular - Growth


In [24]:
top10_sharpe = (
    sharpe_df
    .sort_values(
        "sharpe_ratio",
        ascending=False
    )
    .head(10)
)

top10_sharpe[
    ["scheme_name", "sharpe_ratio"]
]

,scheme_name,sharpe_ratio
27,ICICI Pru Liquid Fund - Regular - Growth,13.524344
31,Kotak Liquid Fund - Regular - Growth,12.447365
5,ABSL Liquid Fund - Regular - Growth,11.890847
34,Mirae Asset Large Cap Fund - Regular - Growth,1.901661
30,Kotak Flexicap Fund - Regular - Growth,1.711792
19,SBI Bluechip Fund - Regular Plan - Growth,1.676558
36,Mirae Asset Tax Saver Fund - Regular - Growth,1.599024
9,Nippon India Large Cap Fund - Regular - Growth,1.536482
25,ICICI Pru Midcap Fund - Regular - Growth,1.513677
38,DSP Midcap Fund - Regular - Growth,1.494736


In [25]:
import plotly.express as px

fig = px.bar(
    top10_sharpe,
    x="sharpe_ratio",
    y="scheme_name",
    orientation="h",
    title="Top 10 Funds by Sharpe Ratio"
)

fig.show()

In [26]:
fig = px.histogram(
    sharpe_df,
    x="sharpe_ratio",
    nbins=20,
    title="Distribution of Sharpe Ratios"
)

fig.show()

In [27]:
sharpe_df.to_csv(
    "../reports/sharpe_values.csv",
    index=False
)

### Sharpe Ratio Interpretation

Sharpe Ratio measures risk-adjusted returns.

General Guidelines:

- Sharpe < 0 → Poor
- 0–1 → Acceptable
- 1–2 → Good
- > 2 → Excellent

Higher values indicate superior returns relative to volatility.

Step 5 — Sortino Ratio

The Sharpe Ratio penalizes all volatility (both upside and downside).

The Sortino Ratio only penalizes bad volatility (negative returns).

Because investors usually don't mind upside volatility, Sortino is often considered a better metric.

In [28]:
RF = 0.065

sortino_results = []

for amfi_code, group in nav.groupby("amfi_code"):

    returns = group["daily_return_pct"].dropna()

    if len(returns) < 30:
        continue

    # Annualized Return
    annual_return = returns.mean() * 252

    # Negative returns only
    downside_returns = returns[returns < 0]

    if len(downside_returns) < 2:
        continue

    downside_deviation = (
        downside_returns.std()
        * np.sqrt(252)
    )

    sortino_ratio = (
        annual_return - RF
    ) / downside_deviation

    sortino_results.append({

        "amfi_code": amfi_code,

        "annual_return": annual_return,

        "downside_deviation": downside_deviation,

        "sortino_ratio": sortino_ratio

    })

sortino_df = pd.DataFrame(sortino_results)

In [29]:
sortino_df = sortino_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

sortino_df.head()

,amfi_code,annual_return,downside_deviation,sortino_ratio,scheme_name
0,100016,3.568303,8.351285,0.419493,HDFC Top 100 Fund - Regular Plan - Growth
1,100025,4.285356,2.351449,1.794790,HDFC Short Term Debt Fund - Regular - Growth
2,100033,27.211057,11.322876,2.397452,HDFC Mid-Cap Opportunities Fund - Regular - Gr...
3,101206,21.464662,8.315721,2.573398,ABSL Frontline Equity Fund - Regular - Growth
4,101207,10.696212,15.168286,0.700884,ABSL Small Cap Fund - Regular - Growth


In [30]:
top10_sortino = (
    sortino_df
    .sort_values(
        "sortino_ratio",
        ascending=False
    )
    .head(10)
)

top10_sortino[
    ["scheme_name", "sortino_ratio"]
]

,scheme_name,sortino_ratio
27,ICICI Pru Liquid Fund - Regular - Growth,28.704082
31,Kotak Liquid Fund - Regular - Growth,26.329334
5,ABSL Liquid Fund - Regular - Growth,24.509274
34,Mirae Asset Large Cap Fund - Regular - Growth,3.132441
30,Kotak Flexicap Fund - Regular - Growth,3.097183
19,SBI Bluechip Fund - Regular Plan - Growth,2.969777
36,Mirae Asset Tax Saver Fund - Regular - Growth,2.779888
9,Nippon India Large Cap Fund - Regular - Growth,2.628089
25,ICICI Pru Midcap Fund - Regular - Growth,2.602985
24,ICICI Pru Bluechip Fund - Direct - Growth,2.593171


In [31]:
import plotly.express as px

fig = px.bar(
    top10_sortino,
    x="sortino_ratio",
    y="scheme_name",
    orientation="h",
    title="Top 10 Funds by Sortino Ratio"
)

fig.show()

In [32]:
comparison = (
    sharpe_df[
        ["amfi_code", "sharpe_ratio"]
    ]
    .merge(
        sortino_df[
            ["amfi_code", "sortino_ratio"]
        ],
        on="amfi_code"
    )
)

In [33]:
fig = px.scatter(
    comparison,
    x="sharpe_ratio",
    y="sortino_ratio",
    title="Sharpe vs Sortino Ratio"
)

fig.show()

In [34]:
sortino_df.to_csv(
    "../reports/sortino_values.csv",
    index=False
)

### Sortino Ratio Interpretation

Sortino Ratio measures return per unit of downside risk.

Unlike Sharpe Ratio, it ignores upside volatility.

General Guidelines:

- Sortino < 0 → Poor
- 0–1 → Average
- 1–2 → Good
- > 2 → Excellent

A fund with a high Sortino Ratio has generated strong returns while minimizing downside risk.

In [37]:
perf.columns.tolist()
perf.head()

,perf_id,amfi_code,as_of_date,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,morningstar_rating
0,1,119551,2026-05-31,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,4
1,2,119552,2026-05-31,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,3
2,3,119598,2026-05-31,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,5
3,4,119599,2026-05-31,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,4
4,5,119120,2026-05-31,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,5


Step 6 — Build Alpha/Beta Dataset

Since Alpha & Beta already exist:

In [38]:
alpha_beta_df = perf[
    [
        "amfi_code",
        "alpha",
        "beta"
    ]
].copy()

alpha_beta_df.head()

,amfi_code,alpha,beta
0,119551,0.87,0.89
1,119552,1.78,0.87
2,119598,1.23,0.89
3,119599,1.13,1.04
4,119120,1.60,0.22


In [39]:
alpha_beta_df = alpha_beta_df.merge(
    funds[
        ["amfi_code", "scheme_name"]
    ],
    on="amfi_code",
    how="left"
)

alpha_beta_df.head()

,amfi_code,alpha,beta,scheme_name
0,119551,0.87,0.89,SBI Bluechip Fund - Regular Plan - Growth
1,119552,1.78,0.87,SBI Bluechip Fund - Direct Plan - Growth
2,119598,1.23,0.89,SBI Small Cap Fund - Regular Plan - Growth
3,119599,1.13,1.04,SBI Small Cap Fund - Direct Plan - Growth
4,119120,1.60,0.22,SBI Magnum Gilt Fund - Regular Plan - Growth


In [40]:
top_alpha = (
    alpha_beta_df
    .sort_values(
        "alpha",
        ascending=False
    )
    .head(10)
)

top_alpha[
    ["scheme_name", "alpha"]
]

,scheme_name,alpha
9,HDFC Short Term Debt Fund - Regular - Growth,1.98
21,Kotak Emerging Equity Fund - Regular - Growth,1.91
14,ICICI Pru Liquid Fund - Regular - Growth,1.85
22,Kotak Flexicap Fund - Regular - Growth,1.85
29,ABSL Small Cap Fund - Regular - Growth,1.84
37,DSP Top 100 Equity Fund - Regular - Growth,1.82
18,Nippon India ETF Nifty 50 BeES,1.80
33,UTI Flexi Cap Fund - Regular - Growth,1.79
1,SBI Bluechip Fund - Direct Plan - Growth,1.78
35,Mirae Asset Emerging Bluechip Fund - Regular -...,1.70


In [41]:
import plotly.express as px

fig = px.bar(
    top_alpha,
    x="alpha",
    y="scheme_name",
    orientation="h",
    title="Top 10 Funds by Alpha"
)

fig.show()

In [42]:
fig = px.scatter(
    alpha_beta_df,
    x="beta",
    y="alpha",
    hover_name="scheme_name",
    title="Alpha vs Beta Analysis"
)

fig.show()

In [43]:
alpha_beta_df.to_csv(
    "../reports/alpha_beta.csv",
    index=False
)

Step 7 — Fund Scorecard (Most Important Part)

Now we can build the final ranking model using official metrics.

In [45]:
scorecard = perf.merge(

    funds[
        [
            "amfi_code",
            "scheme_name",
            "category",
            "expense_ratio_pct"
        ]
    ],

    on="amfi_code",
    how="left"

)

scorecard.head()

,perf_id,amfi_code,as_of_date,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,morningstar_rating,scheme_name,category,expense_ratio_pct
0,1,119551,2026-05-31,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,4,SBI Bluechip Fund - Regular Plan - Growth,Equity,1.54
1,2,119552,2026-05-31,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,3,SBI Bluechip Fund - Direct Plan - Growth,Equity,0.66
2,3,119598,2026-05-31,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,5,SBI Small Cap Fund - Regular Plan - Growth,Equity,1.43
3,4,119599,2026-05-31,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,4,SBI Small Cap Fund - Direct Plan - Growth,Equity,0.72
4,5,119120,2026-05-31,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,5,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,0.77


In [46]:
scorecard["return_rank"] = (
    scorecard["return_3yr_pct"]
    .rank(pct=True)
)

scorecard["sharpe_rank"] = (
    scorecard["sharpe_ratio"]
    .rank(pct=True)
)

scorecard["alpha_rank"] = (
    scorecard["alpha"]
    .rank(pct=True)
)

In [47]:
scorecard["expense_rank"] = (
    (-scorecard["expense_ratio_pct"])
    .rank(pct=True)
)

In [48]:
scorecard["drawdown_rank"] = (
    scorecard["max_drawdown_pct"]
    .rank(
        ascending=False,
        pct=True
    )
)

In [49]:
scorecard["fund_score"] = (

      0.30 * scorecard["return_rank"]

    + 0.25 * scorecard["sharpe_rank"]

    + 0.20 * scorecard["alpha_rank"]

    + 0.15 * scorecard["expense_rank"]

    + 0.10 * scorecard["drawdown_rank"]

) * 100

In [50]:
scorecard = scorecard.sort_values(

    "fund_score",
    ascending=False

)

scorecard["overall_rank"] = range(
    1,
    len(scorecard) + 1
)

In [51]:
scorecard[
    [
        "overall_rank",
        "scheme_name",
        "category",
        "fund_score"
    ]
].head(10)

,overall_rank,scheme_name,category,fund_score
3,1,SBI Small Cap Fund - Direct Plan - Growth,Equity,72.7500
22,2,Kotak Flexicap Fund - Regular - Growth,Equity,71.5000
21,3,Kotak Emerging Equity Fund - Regular - Growth,Equity,70.5000
29,4,ABSL Small Cap Fund - Regular - Growth,Equity,68.2500
2,5,SBI Small Cap Fund - Regular Plan - Growth,Equity,64.3750
34,6,Mirae Asset Large Cap Fund - Regular - Growth,Equity,63.9375
9,7,HDFC Short Term Debt Fund - Regular - Growth,Debt,63.0000
14,8,ICICI Pru Liquid Fund - Regular - Growth,Debt,61.7500
12,9,ICICI Pru Midcap Fund - Regular - Growth,Equity,61.0000
11,10,ICICI Pru Bluechip Fund - Direct - Growth,Equity,59.6250


In [52]:
import plotly.express as px

top10 = scorecard.head(10)

fig = px.bar(

    top10,

    x="fund_score",

    y="scheme_name",

    orientation="h",

    title="Top 10 Mutual Funds by Composite Score"

)

fig.show()

In [53]:
category_winners = (

    scorecard

    .sort_values(
        "fund_score",
        ascending=False
    )

    .groupby("category")

    .head(1)

)

In [54]:
category_winners[
    [
        "category",
        "scheme_name",
        "fund_score"
    ]
]

,category,scheme_name,fund_score
3,Equity,SBI Small Cap Fund - Direct Plan - Growth,72.75
9,Debt,HDFC Short Term Debt Fund - Regular - Growth,63.00


In [55]:
scorecard.to_csv(

    "../reports/fund_scorecard.csv",

    index=False

)

Revised Step 8 — Benchmark Analytics (Proper Version)

In [58]:
benchmark = pd.read_csv(
    "../data/raw/10_benchmark_indices.csv"
)

benchmark["date"] = pd.to_datetime(
    benchmark["date"]
)

benchmark = benchmark.sort_values(
    ["index_name", "date"]
)

benchmark["benchmark_return"] = (

    benchmark
    .groupby("index_name")["close_value"]
    .pct_change()

)

In [59]:
nifty50 = benchmark[
    benchmark["index_name"] == "NIFTY50"
].copy()

nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
].copy()

In [60]:
nav = nav.rename(
    columns={
        "nav_date":"date"
    }
)

In [61]:
fund_returns = nav[
    [
        "amfi_code",
        "date",
        "daily_return_pct"
    ]
].copy()

In [62]:
from scipy.stats import linregress
alpha_beta_results = []

for amfi_code, group in fund_returns.groupby("amfi_code"):

    merged = pd.merge(

        group,

        nifty100[
            [
                "date",
                "benchmark_return"
            ]
        ],

        on="date",

        how="inner"

    ).dropna()

    if len(merged) < 100:
        continue

    slope, intercept, r, p, se = linregress(

        merged["benchmark_return"],

        merged["daily_return_pct"]

    )

    alpha_beta_results.append({

        "amfi_code": amfi_code,

        "alpha": intercept * 252,

        "beta": slope,

        "r_squared": r**2

    })

alpha_beta_real = pd.DataFrame(
    alpha_beta_results
)

In [63]:
alpha_beta_real.to_csv(
    "../reports/alpha_beta.csv",
    index=False
)

In [64]:
tracking_results = []

for amfi_code, group in fund_returns.groupby("amfi_code"):

    merged = pd.merge(

        group,

        nifty100[
            [
                "date",
                "benchmark_return"
            ]
        ],

        on="date"

    ).dropna()

    if len(merged) < 100:
        continue

    tracking_error = (

        (
            merged["daily_return_pct"]

            -

            merged["benchmark_return"]

        ).std()

        * np.sqrt(252)

    )

    tracking_results.append({

        "amfi_code": amfi_code,

        "tracking_error": tracking_error

    })

tracking_df = pd.DataFrame(
    tracking_results
)

In [65]:
top5_codes = (
    scorecard
    .head(5)
    ["amfi_code"]
    .tolist()
)

In [67]:
plot_df = pd.DataFrame()
for code in top5_codes:

    temp = nav[
        nav["amfi_code"] == code
    ].copy()

    temp = temp.sort_values("date")

    temp["normalized"] = (

        temp["nav"]

        /

        temp["nav"].iloc[0]

    ) * 100

    temp["series"] = str(code)

    plot_df = pd.concat(
        [plot_df,temp]
    )

In [68]:
nifty_plot = nifty50.copy()

nifty_plot["normalized"] = (

    nifty_plot["close_value"]

    /

    nifty_plot["close_value"].iloc[0]

) * 100

nifty_plot["series"] = "NIFTY50"

In [69]:
plot_df = pd.concat(
    [plot_df,nifty_plot]
)

In [70]:
import plotly.express as px

fig = px.line(

    plot_df,

    x="date",

    y="normalized",

    color="series",

    title="Top 5 Funds vs NIFTY50"

)

fig.show()

In [71]:
fig.write_image(
    "../reports/benchmark_comparison.png"
)

In [72]:
benchmark_comparison = (
    scorecard[
        [
            "amfi_code",
            "return_3yr_pct"
        ]
    ]
    .merge(
        alpha_beta_real,
        on="amfi_code"
    )
    .merge(
        tracking_df,
        on="amfi_code"
    )
)

In [73]:
benchmark_comparison.to_csv(
    "../reports/benchmark_comparison.csv",
    index=False
)

In [74]:
summary = pd.DataFrame({

    "Metric":[
        "Highest CAGR (3Y)",
        "Highest Sharpe Ratio",
        "Highest Sortino Ratio",
        "Highest Alpha",
        "Lowest Drawdown",
        "Best Overall Fund"
    ],

    "Fund":[

        cagr_df.loc[
            cagr_df["cagr_3yr"].idxmax(),
            "scheme_name"
        ],

        sharpe_df.loc[
            sharpe_df["sharpe_ratio"].idxmax(),
            "scheme_name"
        ],

        sortino_df.loc[
            sortino_df["sortino_ratio"].idxmax(),
            "scheme_name"
        ],

        alpha_beta_real.merge(
            funds[["amfi_code","scheme_name"]],
            on="amfi_code"
        ).loc[
            alpha_beta_real["alpha"].idxmax(),
            "scheme_name"
        ],

        perf.merge(
            funds[["amfi_code","scheme_name"]],
            on="amfi_code"
        ).loc[
            perf["max_drawdown_pct"].idxmax(),
            "scheme_name"
        ],

        scorecard.iloc[0]["scheme_name"]

    ]

})

summary

,Metric,Fund
0,Highest CAGR (3Y),Axis Midcap Fund - Regular - Growth
1,Highest Sharpe Ratio,ICICI Pru Liquid Fund - Regular - Growth
2,Highest Sortino Ratio,ICICI Pru Liquid Fund - Regular - Growth
3,Highest Alpha,SBI Small Cap Fund - Regular Plan - Growth
4,Lowest Drawdown,Nippon India Gilt Securities Fund - Regular - ...
5,Best Overall Fund,SBI Small Cap Fund - Direct Plan - Growth


# Key Findings

1. The highest ranked mutual fund achieved superior risk-adjusted returns, combining strong 3-year performance with favorable Sharpe and Alpha metrics.

2. Several funds significantly outperformed their benchmark indices, generating positive excess returns and strong Alpha values.

3. Tracking Error analysis showed that some actively managed funds deviated substantially from benchmark performance, indicating higher active management.

4. Benchmark comparison revealed that top-ranked funds consistently outperformed NIFTY50 over the analysis period.

5. Maximum Drawdown analysis highlighted the resilience of certain funds during market downturns.